# Wachta: pierwszy kontakt z danymi EEG (Google Colab)

Notebook pobiera dwa otwarte zbiory, pokazuje surowe dane, zawęża je do **naszych 8 kanałów** (F3, F4, C3, C4, P3, P4, O1, O2, 250 Hz) i liczy podstawowe cechy (moc pasm theta, alfa, beta w oknach 2 s). Na końcu sprawdza dwie hipotezy, na których stoi pomysł:

1. **Przeciążenie:** czy theta czołowa (F3/F4) rośnie podczas liczenia w pamięci i trudniejszego zadania?
2. **Senność / oczy zamknięte:** czy alfa potyliczna (O1/O2) rośnie przy zamkniętych oczach?

**Jak uruchomić:** `Środowisko wykonawcze → Uruchom wszystkie`. Całość trwa kilka minut i nic nie trzeba pobierać na laptopa. Dane lecą prosto z archiwów do Colaba, a z dużego zbioru COG-BCI (1,1 GB na osobę) ściągamy tylko kilka potrzebnych plików (ok. 90 MB na osobę).

> Dane są nagrane na laboratoryjnych elektrodach żelowych. Służą do sprawdzenia pipeline'u. Progi do demo muszą pochodzić z kalibracji na BrainAccess MINI.

## 0. Instalacja i konfiguracja

In [ ]:
!pip install -q mne remotezip

In [ ]:
import shutil
import urllib.request
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import mne
import numpy as np
import pandas as pd
from remotezip import RemoteZip
from scipy.stats import wilcoxon

mne.set_log_level("ERROR")
warnings.filterwarnings("ignore")

DATA = Path("/content/data")
OUT = Path("/content/features")
DATA.mkdir(parents=True, exist_ok=True)
OUT.mkdir(parents=True, exist_ok=True)

OUR_CH = ["F3", "F4", "C3", "C4", "P3", "P4", "O1", "O2"]
TARGET_SFREQ = 250
WIN_S, STEP_S = 2.0, 1.0                      # 2 s windows, 1 s step
BANDS = {"theta": (4, 7), "alpha": (8, 12), "beta": (13, 30)}
REJECT_UV = 300                               # drop windows with peak-to-peak above this (blinks, motion)

## 1. Funkcje pomocnicze: obróbka, cechy, wykresy

In [ ]:
def prepare(raw):
    """Keep only our 8 channels, band-pass 1-40 Hz, resample to 250 Hz."""
    raw = raw.copy()
    raw.rename_channels(lambda n: n.replace("EEG ", "").strip())
    raw.set_annotations(None)
    raw.pick(OUR_CH)
    raw.reorder_channels(OUR_CH)
    raw.filter(1.0, 40.0, fir_design="firwin", verbose=False)
    raw.resample(TARGET_SFREQ)
    return raw


def band_features(raw, meta):
    """Band power (log10 uV^2) per 2 s window and channel, plus aggregates."""
    ep = mne.make_fixed_length_epochs(
        raw, duration=WIN_S, overlap=WIN_S - STEP_S, preload=True
    )
    n_all = len(ep)
    ep.drop_bad(reject=dict(eeg=REJECT_UV * 1e-6))
    psd = ep.compute_psd(
        method="welch", fmin=1, fmax=40, n_fft=int(WIN_S * TARGET_SFREQ)
    )
    data, freqs = psd.get_data(return_freqs=True)   # (epochs, ch, freqs), V^2/Hz
    data = data * 1e12                              # -> uV^2/Hz
    df_hz = freqs[1] - freqs[0]

    cols, lin = {}, {}
    for band, (lo, hi) in BANDS.items():
        mask = (freqs >= lo) & (freqs <= hi)
        p = data[:, :, mask].sum(-1) * df_hz        # (epochs, ch), uV^2
        lin[band] = p
        for i, ch in enumerate(OUR_CH):
            cols[f"{band}_{ch}"] = np.log10(p[:, i])
    out = pd.DataFrame(cols)
    out["theta_front"] = out[["theta_F3", "theta_F4"]].mean(axis=1)
    out["alpha_par"] = out[["alpha_P3", "alpha_P4"]].mean(axis=1)
    out["alpha_occ"] = out[["alpha_O1", "alpha_O2"]].mean(axis=1)
    out["engagement"] = lin["beta"].mean(1) / (lin["alpha"].mean(1) + lin["theta"].mean(1))
    out.insert(0, "t_start", ep.events[:, 0] / ep.info["sfreq"])
    for k, v in reversed(list(meta.items())):
        out.insert(0, k, v)
    print(f"  {meta['subject']} {meta['condition']}: {len(ep)}/{n_all} windows kept")
    return out


def plot_traces(raw, tmin=0, dur=10, title=""):
    data = raw.get_data(tmin=tmin, tmax=tmin + dur) * 1e6
    t = np.arange(data.shape[1]) / raw.info["sfreq"] + tmin
    step = np.percentile(np.abs(data), 99) * 2.2
    fig, ax = plt.subplots(figsize=(11, 5))
    for i in range(len(raw.ch_names)):
        ax.plot(t, data[i] - i * step, lw=0.7)
    ax.set_yticks(-np.arange(len(raw.ch_names)) * step, raw.ch_names)
    ax.set_xlabel("czas [s]")
    ax.set_title(title)
    plt.tight_layout()
    plt.show()


def plot_psd(raws, chs, fmax=40):
    fig, axes = plt.subplots(1, len(chs), figsize=(11, 3.5), sharey=True)
    for ax, ch in zip(np.atleast_1d(axes), chs):
        for name, r in raws.items():
            p = r.compute_psd(method="welch", fmax=fmax, n_fft=1000, picks=[ch])
            d, f = p.get_data(return_freqs=True)
            ax.semilogy(f, d[0] * 1e12, label=name)
        ax.axvspan(8, 12, color="gold", alpha=0.15)
        ax.set_title(ch)
        ax.set_xlabel("Hz")
    np.atleast_1d(axes)[0].set_ylabel("uV^2/Hz")
    np.atleast_1d(axes)[0].legend()
    plt.tight_layout()
    plt.show()


def paired_plot(df, col, a, b, ax):
    """Per-subject mean of `col` in condition a vs b, with a Wilcoxon test."""
    m = df.groupby(["subject", "condition"])[col].mean().unstack()[[a, b]].dropna()
    for _, r in m.iterrows():
        ax.plot([0, 1], [r[a], r[b]], color="gray", alpha=0.5, marker="o", ms=3)
    ax.set_xticks([0, 1], [a, b])
    p = wilcoxon(m[a], m[b]).pvalue
    up = (m[b] > m[a]).mean() * 100
    ax.set_title(f"{col}\n{up:.0f}% osób w górę, Wilcoxon p={p:.3g}, n={len(m)}")

## 2. Zbiór A: Mental Arithmetic (PhysioNet), spoczynek vs liczenie w pamięci

36 osób, po dwa pliki: `_1` to spoczynek, `_2` to liczenie. Pliki są małe (kilka MB), więc pobieramy wszystkie. Zmniejsz `N_EEGMAT`, jeśli chcesz szybciej.

In [ ]:
N_EEGMAT = 36
EEGMAT_URL = "https://physionet.org/files/eegmat/1.0.0"


def download_eegmat(i):
    folder = DATA / "eegmat"
    folder.mkdir(exist_ok=True)
    for part in (1, 2):
        f = folder / f"Subject{i:02d}_{part}.edf"
        if not f.exists():
            req = urllib.request.Request(
                f"{EEGMAT_URL}/{f.name}", headers={"User-Agent": "Mozilla/5.0"}
            )
            with urllib.request.urlopen(req) as r, open(f, "wb") as out:
                shutil.copyfileobj(r, out)


for i in range(N_EEGMAT):
    download_eegmat(i)
print("Pobrano:", len(list((DATA / "eegmat").glob("*.edf"))), "plików")

### Podgląd surowych danych (osoba 0)

In [ ]:
raw0 = mne.io.read_raw_edf(DATA / "eegmat" / "Subject00_1.edf", preload=True)
print(raw0)
print("Kanały w pliku:", raw0.ch_names)
print("Czas nagrania [s]:", round(raw0.times[-1], 1), "| próbkowanie:", raw0.info["sfreq"], "Hz")

prep0 = prepare(raw0)
print("Po obróbce:", prep0.ch_names, prep0.info["sfreq"], "Hz")
plot_traces(prep0, tmin=20, dur=10, title="Mental Arithmetic, osoba 0, spoczynek (20-30 s), nasze 8 kanałów")

### Cechy dla wszystkich osób

In [ ]:
frames = []
for i in range(N_EEGMAT):
    for part, cond in ((1, "rest"), (2, "task")):
        path = DATA / "eegmat" / f"Subject{i:02d}_{part}.edf"
        raw = prepare(mne.io.read_raw_edf(path, preload=True))
        frames.append(
            band_features(raw, dict(dataset="eegmat", subject=f"S{i:02d}", condition=cond))
        )
eegmat = pd.concat(frames, ignore_index=True)
eegmat.head()

### Test hipotezy: theta czołowa i alfa ciemieniowa, spoczynek vs liczenie

Spodziewamy się theta czołowej w górę i alfy ciemieniowej w dół przy liczeniu. Każda linia to jedna osoba (średnia ze wszystkich jej okien).

**Co wyszło w moim teście na 12 osobach** (u Ciebie może się nieco różnić): alfa ciemieniowa spadała u ok. 90% osób (p ≈ 0,01), zgodnie z oczekiwaniem. Theta czołowa **nie** rosła wyraźnie, tylko u ok. 40% osób w górę. To zgadza się z założeniem w opisie pomysłu, że przeciążenie jest mniej stabilne niż senność, i jest pierwszą rzeczą do zbadania (inne okna, inne pasmo, osobna obsługa mrugnięć).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, ["theta_front", "alpha_par", "engagement"]):
    paired_plot(eegmat, col, "rest", "task", ax)
plt.tight_layout()
plt.show()

### Wskaźnik w stylu Wachty: z-score względem własnej bazy

Tak będzie działać demo: 60 s kalibracji, a potem każde okno liczymy jako odchylenie od bazy tej osoby. Tu bazą jest spoczynek.

In [ ]:
def zscore_vs_baseline(df, col, base_cond):
    base = df[df.condition == base_cond].groupby("subject")[col].agg(["mean", "std"])
    z = (df[col] - df.subject.map(base["mean"])) / df.subject.map(base["std"])
    return z


eegmat["z_theta_front"] = zscore_vs_baseline(eegmat, "theta_front", "rest")
eegmat["z_alpha_par"] = zscore_vs_baseline(eegmat, "alpha_par", "rest")

print(eegmat.groupby("condition")[["z_theta_front", "z_alpha_par"]].agg(["mean", "std"]).round(2))

fig, ax = plt.subplots(figsize=(8, 3.5))
for cond in ("rest", "task"):
    ax.hist(eegmat[eegmat.condition == cond].z_theta_front.clip(-4, 6),
            bins=40, alpha=0.5, density=True, label=cond)
ax.set_xlabel("z-score theta czołowej (F3/F4) względem własnego spoczynku")
ax.legend()
plt.show()

## 3. Zbiór B: COG-BCI, najbliżej scenariusza operatora

Archiwum jednej osoby waży 1,1 GB, ale czytamy je **zdalnie po kawałku** (`remotezip`) i wyciągamy tylko cztery potrzebne nagrania z sesji pierwszej:

| Plik | Co to jest | Czas |
|---|---|---|
| `RS_Beg_EC` / `RS_Beg_EO` | spoczynek, oczy zamknięte / otwarte | 60 s |
| `MATBeasy` / `MATBmed` / `MATBdiff` | symulacja pracy operatora NASA MATB-II, trzy poziomy trudności | 5 min |

Razem ok. 100 MB na osobę. Zbiór ma wszystkie nasze 8 kanałów (500 Hz, 63 kanały w pliku). Nie każda osoba musi mieć każdy plik, brakujące są pomijane.

In [ ]:
COG_URL = "https://zenodo.org/records/7413650/files/{sub}.zip?download=1"
COG_SUBJECTS = [f"sub-{i:02d}" for i in range(1, 9)]   # 8 osób, max sub-29
COG_SESSION = "ses-S1"
COG_FILES = {
    "RS_Beg_EC": "rest_eyes_closed",
    "RS_Beg_EO": "rest_eyes_open",
    "MATBeasy": "matb_easy",
    "MATBmed": "matb_medium",
    "MATBdiff": "matb_difficult",
}

frames, preview = [], {}
for sub in COG_SUBJECTS:
    try:
        with RemoteZip(COG_URL.format(sub=sub)) as z:
            names = z.namelist()
            # folder layout differs between subjects (e.g. sub-03/sub-03/...), so match by suffix
            to_get = [n for n in names
                      if any(n.endswith(f"/{COG_SESSION}/eeg/{stem}{ext}")
                             for stem in COG_FILES for ext in (".set", ".fdt"))]
            for n in to_get:
                z.extract(n, DATA / "cogbci" / sub)
    except Exception as e:
        print("pomijam", sub, "-", e)
        continue
    if not to_get:
        print("pomijam", sub, "- brak plików dla", COG_SESSION)
        continue
    for stem, cond in COG_FILES.items():
        found = list((DATA / "cogbci" / sub).rglob(f"{COG_SESSION}/eeg/{stem}.set"))
        if not found:
            continue
        raw = prepare(mne.io.read_raw_eeglab(found[0], preload=True))
        preview.setdefault(cond, raw)
        frames.append(band_features(raw, dict(dataset="cogbci", subject=sub, condition=cond)))
    shutil.rmtree(DATA / "cogbci" / sub, ignore_errors=True)   # free disk

cog = pd.concat(frames, ignore_index=True)
cog.head()

### Podgląd: surowy zapis i widmo (oczy zamknięte vs otwarte)

In [ ]:
plot_traces(preview["rest_eyes_closed"], tmin=10, dur=10,
            title=f"COG-BCI {COG_SUBJECTS[0]}, oczy zamknięte, nasze 8 kanałów")
plot_psd({"oczy zamknięte": preview["rest_eyes_closed"],
          "oczy otwarte": preview["rest_eyes_open"]}, chs=["O1", "O2", "F3"])

### Test hipotez

- **Alfa potyliczna:** oczy zamknięte vs otwarte. To najpewniejszy sygnał, powinien wyjść wyraźnie.
- **Obciążenie:** łatwe vs trudne zadanie MATB. Tu efekt jest słabszy i mniej stabilny, zgodnie z założeniem w opisie pomysłu.

W moim teście na 4 osobach alfa potyliczna rosła przy zamkniętych oczach u wszystkich (wyraźne wrzeciona alfy widać już na surowym zapisie), a theta czołowa rosła w trudnym zadaniu MATB u wszystkich. Przy tak małej próbie test Wilcoxona nie zejdzie poniżej p = 0,125, dlatego domyślnie bierzemy 8 osób.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
paired_plot(cog, "alpha_occ", "rest_eyes_open", "rest_eyes_closed", axes[0])
paired_plot(cog, "theta_front", "matb_easy", "matb_difficult", axes[1])
paired_plot(cog, "alpha_par", "matb_easy", "matb_difficult", axes[2])
plt.tight_layout()
plt.show()

## 4. Zapis cech

In [ ]:
features = pd.concat([eegmat, cog], ignore_index=True)
path = OUT / "features.csv.gz"
features.to_csv(path, index=False)
print(features.shape, "->", path, f"({path.stat().st_size / 1e6:.1f} MB)")
features.groupby(["dataset", "condition"]).size()

Pobranie pliku na komputer (wynik to kilka MB, nie surowe dane):

In [ ]:
from google.colab import files
files.download(str(path))

Opcjonalnie zapis na Dysk Google, żeby zespół miał wspólny dostęp (odkomentuj):

In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
# shutil.copy(path, "/content/drive/MyDrive/heroes-of-the-brain/features.csv.gz")

## Na co uważać

- **Inny sprzęt:** elektrody żelowe, inny montaż odniesienia (EEGMAT: uszy, COG-BCI: inna referencja), więc wartości bezwzględne nie przenoszą się na BrainAccess. Przenosi się kierunek zmian i sposób liczenia cech.
- **Brak ICA:** EEGMAT jest już oczyszczony z artefaktów, COG-BCI nie. Okna z amplitudą (peak-to-peak) powyżej 300 µV są odrzucane. To luźny próg, bo w COG-BCI typowe okna mają 100-200 µV. Mruganie i ruchy gałek wpływają na theta czołową, więc przy BrainAccess warto je wykrywać osobno.
- **To są średnie po osobach.** W demo liczy się stabilność u jednej osoby po 60 s kalibracji, więc warto sprawdzić też rozrzut wewnątrz osoby (histogram z-score wyżej).
- **Licencje:** PhysioNet (ODC-By 1.0) i COG-BCI wymagają podania źródła w prezentacji.